# Feature selection by category: Patient events

Finds the strongest **patient event** features (Rx / Dx / Px / lab first occurrence, frequency, recency, hospitalization and the add-on patient features), separately for **existing writers** and **non-writers**, on the 10k-HCP sample with all 28 cohorts. Reads only this category's 15 source tables and fills empty values with the Combine rules.

Same method as every category notebook: the category config (cell 3) is the only difference; all steps run from [00_category_feature_selection_core](./00_category_feature_selection_core) via `%run` (cell 4). **Part 1:** writer definition (`TOTAL_HCP_NBRX_CLAIMS_KERENDIA_IN_LAST_12_MONTH` > 0, empty = 0), writers / non-writers data sets, mean / median analysis (`DISCRIMINATION` = \|2 x AUC - 1\|). **Part 2:** NBA 3.1 clean-up (stages 1-4, flags 5-6), one over-fitted LightGBM Poisson model per group on all 28 cohorts, SHAP, best features. Outputs: `feature_selection_by_category/<RUN_NAME>/patient_events/`.

In [0]:
%pip install lightgbm==4.5.0 --quiet
dbutils.library.restartPython()

In [0]:
CATEGORY       = "patient_events"
CATEGORY_LABEL = "Patient events: Rx / Dx / Px / lab first occurrence, frequency, recency, hospitalization, add-on"
ADD_CALENDAR   = False

VOL31 = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/"
NO    = VOL31 + "notebook_output/"
DEV   = VOL31 + "hcp_level_model/dev_pipeline/notebook_output/"

def _t(name, sub, path, fmt, cohort_col, drop_cols=()):
    """One source table, defined as in NBA_3_1_Combine_Physician_Data (cell 2)."""
    return dict(name=name, sub_category=sub, path=path, fmt=fmt, cohort_col=cohort_col, drop_cols=list(drop_cols))

# Patient events (15 tables, as in Combine)
SOURCE_TABLES = []
for fam in ["rx", "dx", "px", "lab"]:
    kinds = ["first_occurrence", "frequency", "recency"] + (["hospitalization"] if fam in ("dx", "px") else [])
    for k in kinds:
        SOURCE_TABLES.append(_t(f"{fam}_{k}_features", f"{fam.upper()} {k.replace('_', ' ')}",
                                f"{NO}patient_features/{fam}/{fam}_{k}_features", "delta", "COHORT"))
SOURCE_TABLES.append(_t("claims_addon_patient_features", "Add-on", NO + "patient_features/addon/claims_addon_patient_features", "delta", "COHORT"))

In [0]:
%run ./00_category_feature_selection_core